# Finetune F5-TTS-Vietnamese tren Colab

**Truoc khi chay**, upload 4 file len Google Drive tai `MyDrive/f5tts/`:

| File | Nguon tren may Mac | Kich thuoc |
|---|---|---|
| `pretrained_vn1000h.pt` | `f5tts/F5-TTS-Vietnamese/ckpts/your_training_dataset/` | 1.3 GB |
| `your_training_dataset.zip` | nen tu `f5tts/F5-TTS-Vietnamese/data/your_training_dataset/` | ~117 MB |
| `colab.patch` | `f5tts/` | 3 KB |
| `run_colab.sh` | `f5tts/` | 1 KB |

Lenh tao zip tren may Mac:

```bash
cd f5tts/F5-TTS-Vietnamese/data && zip -r ~/your_training_dataset.zip your_training_dataset
```

Nho chon **Runtime > Change runtime type > T4 GPU** truoc khi chay.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '| torch:', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/f5tts'
!ls -lh {DRIVE}

In [ ]:
%cd /content
!rm -rf F5-TTS-Vietnamese
!git clone https://github.com/nguyenthienhy/F5-TTS-Vietnamese
%cd /content/F5-TTS-Vietnamese
!git checkout e74db9d
!git apply {DRIVE}/colab.patch
!git diff --stat

Cai deps bang `--no-deps` de **khong dung toi torch cua Colab**. Cai nguyen khoi `pip install -e .` se keo ve torch khac va `numpy<=1.26.4`, buoc phai restart runtime.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q accelerate cached_path click datasets ema_pytorch hydra-core jieba \
    librosa pydub pypinyin safetensors soundfile tomli torchdiffeq transformers \
    vocos x_transformers bitsandbytes

In [ ]:
!mkdir -p data ckpts/your_training_dataset
!unzip -q -o {DRIVE}/your_training_dataset.zip -d data/
!cp {DRIVE}/pretrained_vn1000h.pt ckpts/your_training_dataset/
!cp {DRIVE}/run_colab.sh /content/ && chmod +x /content/run_colab.sh
!ls data/your_training_dataset | head
!echo "wav: $(ls data/your_training_dataset/wavs | wc -l)"

Kiem tra vocab khop voi embedding cua pretrained. Neu lech, training se bao loi shape khi load — **khong duoc chay `check_vocab_pretrained.py`**, no se ghi de `vocab.txt`.

In [ ]:
import torch

vocab = sum(1 for _ in open('data/your_training_dataset/vocab.txt', encoding='utf-8'))
ckpt = torch.load('ckpts/your_training_dataset/pretrained_vn1000h.pt', map_location='cpu', weights_only=False)
state = ckpt.get('ema_model_state_dict') or ckpt.get('model_state_dict')
rows = next(v.shape[0] for k, v in state.items() if k.endswith('text_embed.text_embed.weight'))
print(f'vocab.txt = {vocab} dong | text_embed = {rows} hang')
print('KHOP' if rows == vocab + 1 else 'LECH - dung lai, kiem tra lai vocab')
del ckpt, state

## Train

Neu bao **CUDA out of memory**, ha `BATCH_FRAMES` xuong 6000 roi 4000, hoac them `--bnb_optimizer` vao `run_colab.sh` de dung Adam 8-bit (tiet kiem ~2.7 GB).

Checkpoint ghi vao `/content` (dia local, nhanh). Cell ke tiep moi copy sang Drive.

In [ ]:
%cd /content/F5-TTS-Vietnamese
!REPO_DIR=/content/F5-TTS-Vietnamese EPOCHS=35 bash /content/run_colab.sh

## Rut trong so cuoi cung

Trainer luu trong so cuoi (sau khi LR da giam ve ~0) vao `model_last.pt` chu khong thanh file danh so — day moi la checkpoint dang gia nhat. Cell duoi rut phan EMA ra thanh `model_final.pt` 1.3 GB, cung dinh dang voi cac file `model_<update>.pt`.

In [ ]:
import torch, os

SRC = 'ckpts/your_training_dataset/model_last.pt'
DST = 'ckpts/your_training_dataset/model_final.pt'

full = torch.load(SRC, map_location='cpu', weights_only=False)
print('update cuoi:', full.get('update'))
print('cac key   :', list(full.keys()))

torch.save({'ema_model_state_dict': full['ema_model_state_dict']}, DST)
del full

print(f'{os.path.getsize(SRC)/1e9:.2f} GB -> {os.path.getsize(DST)/1e9:.2f} GB')

In [ ]:
!mkdir -p {DRIVE}/ckpts
!ls -lh ckpts/your_training_dataset/
!cp ckpts/your_training_dataset/model_final.pt {DRIVE}/ckpts/
!cp -r ckpts/your_training_dataset/samples {DRIVE}/ckpts/
!du -sh {DRIVE}/ckpts

**Dung copy `model_*.pt` bang wildcard.** Voi `save_per_updates 200`, mot lan chay 35 epoch sinh ra hang chuc checkpoint 1.3 GB — copy het se vuot Drive free 15 GB. Chi copy `model_final.pt`; muon them mot moc doi chieu thi them tay, vi du `model_2400.pt`.

`model_last.pt` (5.4 GB) chi can neu ban dinh resume train sau nay.

## Nghe thu — tieu chi dung

Sinh cung mot cau bang `pretrained_vn1000h.pt` va `model_final.pt`, cung ref. Neu ban finetune **khong hon** ban pretrained mot cach nghe thay duoc, thi dung finetune va chuyen sang tang du lieu — dung train them.

`gen_text` bat buoc viet thuong: chu hoa tieng Viet co dau (`Ứ`, `Ơ`, `Ế`...) khong co trong vocab va se bi nuot.

In [ ]:
from IPython.display import Audio, display
import os

REF = 'data/your_training_dataset/wavs/sample_00004.wav'
REF_TEXT = 'tình cờ lạc bước vào một ngôi mộ hoang của quý phi đời trước.'
GEN_TEXT = ('phương ứng vật liền yên tâm, không phải để bản thân thực sự học lại từ đầu '
            'một lượt là được, kiểu đó ba bốn tháng thời gian tuyệt đối chẳng đủ dùng, '
            'bản thân lại chẳng có được cái tài nhìn qua là nhớ.')

os.makedirs('/content/thu', exist_ok=True)

for ckpt in ['pretrained_vn1000h.pt', 'model_final.pt']:
    out = ckpt.replace('.pt', '.wav')
    !f5-tts_infer-cli --model F5TTS_Base \
        --ref_audio {REF} --ref_text "{REF_TEXT}" --gen_text "{GEN_TEXT}" \
        --speed 0.9 --vocoder_name vocos \
        --vocab_file data/your_training_dataset/vocab.txt \
        --ckpt_file ckpts/your_training_dataset/{ckpt} \
        --output_dir /content/thu --output_file {out} > /dev/null 2>&1
    print(ckpt)
    display(Audio(f'/content/thu/{out}'))